In [1]:
import json
import os
from typing import  Union
from pathlib import Path

import pandas as pd


In [2]:
def load_dna_json(dna_path: str):
    """Load DNA data from .json (list) or .jsonl (one JSON object per line)."""
    if dna_path.endswith(".jsonl"):
        records = []
        with open(dna_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    records.append(json.loads(line))
        return records
    else:
        # .json
        with open(dna_path, "rb") as f:
            return json.load(f)  # expected to be a list


def get_all_dna_json_paths(dna_path: Union[str, os.PathLike]):
    """Return all DNA JSON/JSONL file paths in the directory."""
    all_files = [f for f in os.listdir(dna_path) if f.endswith(".json") or f.endswith(".jsonl")]
    if not all_files:
        raise FileNotFoundError(f'No DNA JSON/JSONL files found in {dna_path}')
    # Prefer .jsonl order first (optional), then .json
    all_files.sort(key=lambda x: (not x.endswith(".jsonl"), x))
    return [os.path.join(dna_path, f) for f in all_files]

In [3]:
path = (Path(os.getcwd())).parent / 'Data' / 'fine_tuning'
path

WindowsPath('C:/Users/minem/PycharmProjects/Master/GroovePal/Data/fine_tuning')

In [4]:
train_paths = get_all_dna_json_paths(path / 'train')
test_paths = get_all_dna_json_paths(path / 'test')
valid_paths = get_all_dna_json_paths(path / 'validation')

paths = train_paths + test_paths + valid_paths
paths

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'C:\\Users\\minem\\PycharmProjects\\Master\\GroovePal\\Data\\fine_tuning\\test'

In [5]:
dnas = []
for json_path in paths:
    dna = load_dna_json(json_path)
    dnas.extend(dna)

In [6]:
len(dnas)

20899

In [7]:
df = pd.DataFrame(dnas)
df

,DNA_ID,StyleTags,DNAUnits,Bpm,GridFactor,Numerator,Denominator,TicksPerQuarterNote,NumberOfBars,TicksPerGridUnit,FillStartTicks
0,hs_10,"[Backbeat Ride, Jam1 Hard Rock T120 (0-860)]","[{'Value': 0, 'ExcludeValue': None, 'Wildcard'...",120,4,4,4,480,4,120,0
1,wb_100,"[Halftime HH, Jam1 Rock T120 (1-928)]","[{'Value': 0, 'ExcludeValue': None, 'Wildcard'...",120,4,4,4,480,6,120,0
2,wb_1003,"[Backbeat Ride, Jam2 Disco T125 (971 - 1544)]","[{'Value': 0, 'ExcludeValue': None, 'Wildcard'...",125,4,4,4,480,4,120,0
3,hs_1005,"[Backbeat HH, Jam2 Ballad Grooves T95 (980 - 1...","[{'Value': 0, 'ExcludeValue': None, 'Wildcard'...",95,4,4,4,480,4,120,0
4,wb_1007,"[Backbeat Ride, Jam2 Disco T125 (971 - 1544)]","[{'Value': 0, 'ExcludeValue': None, 'Wildcard'...",125,4,4,4,480,4,120,0
...,...,...,...,...,...,...,...,...,...,...,...
20894,foundational_9956,[FILLS],"[{'Value': 2, 'ExcludeValue': None, 'Wildcard'...",190,4,6,8,9600,0,4800,0
20895,foundational_9980,[PRE_CHORUS],"[{'Value': 5, 'ExcludeValue': None, 'Wildcard'...",140,4,4,4,9600,8,2400,0
20896,foundational_9987,[CHORUS],"[{'Value': 5, 'ExcludeValue': None, 'Wildcard'...",140,4,4,4,9600,8,2400,0
20897,foundational_9990,[BRIDGE],"[{'Value': 5, 'ExcludeValue': None, 'Wildcard'...",140,4,4,4,9600,8,2400,0


In [8]:
df['Unit_lengths'] = df['DNAUnits'].apply(len)
df['Unit_lengths']

0         79
1        109
2         79
3         79
4         79
        ... 
20894     12
20895    128
20896    128
20897    128
20898     32
Name: Unit_lengths, Length: 20899, dtype: int64

In [9]:
less_than_1024 = df['Unit_lengths'] < 1024
less_than_1024

0        True
1        True
2        True
3        True
4        True
         ... 
20894    True
20895    True
20896    True
20897    True
20898    True
Name: Unit_lengths, Length: 20899, dtype: bool

In [10]:
units = [unit for lst_units in df["DNAUnits"] for unit in lst_units]
units = pd.DataFrame(units)

In [11]:
units

,Value,ExcludeValue,Wildcard,AvgOffsetTicks,AvgVelocity,OffsetTicksPerValuePart,VelocityPerValuePart,IsEmpty
0,0,None,False,0,0.000000,{},{},True
1,0,None,False,0,0.000000,{},{},True
2,0,None,False,0,0.000000,{},{},True
3,0,None,False,0,0.000000,{},{},True
4,0,None,False,0,0.000000,{},{},True
...,...,...,...,...,...,...,...,...
1776003,2,None,False,-140,0.535433,{'2': -140},{'2': 0.5354330708661417},False
1776004,2,None,False,-180,0.968504,{'2': -180},{'2': 0.968503937007874},False
1776005,2,None,False,-240,0.968504,{'2': -240},{'2': 0.968503937007874},False
1776006,2,None,False,-120,0.905512,{'2': -120},{'2': 0.905511811023622},False
